---
title: Understanding Neighborhood Enrichment Analysis with Squidpy
description: What Squidpy's neighborhood enrichment score actually computes on a Xenium dataset, and how to read the result.
author: "Axel R"
date: "today"
categories:
  - Spatial Transcriptomics
  - Xenium
toc: true
code-annotations: select
number-sections: true
lightbox: true
title-block-banner: true
image: "spatial_neigh.png"
---

::: {.callout-warning appearance="default" title="Work in progress"}
This site is still under development, so parts of it are rough. Check back for updates.
:::

# Why this notebook

Spatial transcriptomics tells you where cells sit in a tissue, not only what they express. [Squidpy](https://squidpy.readthedocs.io/en/stable/index.html) is one of the main Python libraries for analyzing that kind of data (the paper is [here](https://www.nature.com/articles/s41592-021-01358-2)).

This guide covers one function: *neighborhood enrichment analysis*. It scores which pairs of cell types end up next to each other more often than you would expect by chance, which is where you start when asking about cell-cell interactions in a tissue.

The first time I ran it, I could not connect the output to anything biological. So I went digging into what the function computes, and this notebook is what came out of that.

Suggestions and questions are welcome in the comments.

# Download the data

Following the [Squidpy tutorial on analysis of Xenium data](https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/tutorial_xenium.html), we use a [Xenium dataset of human lung cancer](https://www.10xgenomics.com/datasets/preview-data-ffpe-human-lung-cancer-with-xenium-multimodal-cell-segmentation-1-standard).

::: {.callout-note appearance="default" collapse="true" title="How to download this dataset"}
1. Go to the dataset's preview page.
2. Under **Output and supplemental files**, download the **Xenium Output Bundle** (we use the full dataset).
3. Unzip the downloaded file.
4. Inside the unzipped folder, unzip all the `.zip` files as well.

:::

In [2]:
# Load libraries
import spatialdata as sd
from spatialdata_io import xenium

import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import squidpy as sq

/home/axel/miniconda3/envs/ne_analysis/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/axel/miniconda3/envs/ne_analysis/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/home/axel/miniconda3/envs/ne_analysis/lib/python3.12/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_

In [15]:
from pathlib import Path
import os
cwd = Path.cwd()
xenium_path = os.path.join(cwd, "data", "Xenium")